# 07 — MCDA Scenario Analysis & Traceability Report

Evaluate each scenario on 5 criteria (Impact, Probability, Actionability, Time Horizon, Risk Severity).
Apply AHP for criterion weighting and TOPSIS for multi-criteria ranking.
Visualize as MCDA ranking, criteria radar, and Impact vs Probability matrix.
Generate full traceability report for top scenarios.

In [ ]:
import sys, os, json
import matplotlib.pyplot as plt
import numpy as np
sys.path.insert(0, os.path.abspath(".."))

from src.models.scenarios import Scenario
from src.models.evaluation import Assessment
from src.models.drivers import TechDriver
from src.pipeline.evaluation import (
    compute_scenario_confidences, build_scenarios_block, assess_scenarios,
    run_mcda, generate_traceability_report,
)
from src.config import MCDA_CRITERIA, MCDA_PAIRWISE_DEFAULT
from src.llm import embed
from src.rag import get_collection

import chromadb
collection = get_collection()

with open("../data/outputs/scenario_state.json") as f:
    scenarios = [Scenario(**s) for s in json.load(f)["scenarios"]]
with open("../data/outputs/merge_state.json") as f:
    merge_data = json.load(f)
    drivers = [TechDriver(**d) for d in merge_data["unified_drivers"]]
with open("../data/outputs/kb_state.json") as f:
    kb_state = json.load(f)

driver_by_id = {d.id: d for d in drivers}
print(f"Analyzing {len(scenarios)} scenarios across {len(MCDA_CRITERIA)} criteria")

In [ ]:
# LLM-based batch assessment (5 criteria) + MCDA ranking
confidences = compute_scenario_confidences(scenarios, driver_by_id)
assessments = assess_scenarios(scenarios, confidences, collection)
ahp, mcda_results = run_mcda(assessments)

# sort by MCDA rank
ranked = sorted(zip(scenarios, assessments, mcda_results), key=lambda x: x[2].rank)

print(f"AHP Consistency Ratio: {ahp.consistency_ratio:.4f} ({'OK' if ahp.is_consistent else 'WARNING: inconsistent'})\n")
print(f"{'Rank':<5} {'Type':<14} {'Scenario':<60} {'TOPSIS':<8} {'Imp':<5} {'Prob':<5} {'Act':<5} {'Time':<5} {'Risk':<5}")
print("-" * 112)
for scenario, assessment, mcda in ranked:
    print(f"#{mcda.rank:<4} {scenario.type.value:<14} {scenario.title[:58]:<60} "
          f"{mcda.topsis_closeness:.3f}   {assessment.impact:<5.1f}{assessment.probability:<5.1f}"
          f"{assessment.actionability:<5.1f}{assessment.time_horizon:<5.1f}{assessment.risk_severity:<5.1f}")

## MCDA Visualizations

In [ ]:
type_colors = {
    "evolutionary": "#2196F3",
    "disruptive": "#FF5722",
    "cautionary": "#FFC107",
    "wildcard": "#9C27B0",
}

# --- 1. MCDA Ranking Bar Chart ---
fig, ax = plt.subplots(figsize=(12, 5))
ranked_scenarios = sorted(zip(scenarios, mcda_results), key=lambda x: x[1].rank)
names = [s.title[:40] for s, _ in ranked_scenarios]
closeness = [m.topsis_closeness for _, m in ranked_scenarios]
colors = [type_colors.get(s.type.value, "#999") for s, _ in ranked_scenarios]

bars = ax.barh(names[::-1], closeness[::-1], color=colors[::-1], edgecolor="black", linewidth=0.5)
ax.set_xlabel("TOPSIS Closeness Coefficient", fontsize=11)
ax.set_title("MCDA Scenario Ranking (AHP-weighted TOPSIS)", fontsize=12)
ax.set_xlim(0, 1)
for i, (bar, c) in enumerate(zip(bars, closeness[::-1])):
    rank = len(closeness) - i
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"#{rank} ({c:.3f})", va="center", fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, edgecolor="black", label=t.capitalize()) for t, c in type_colors.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("../data/outputs/mcda_ranking.png", dpi=150)
plt.show()

# --- 2. Criteria Radar Chart ---
criteria_labels = ["Impact", "Probability", "Actionability", "Time Horizon", "Risk Severity"]
n_criteria = len(criteria_labels)
angles = np.linspace(0, 2 * np.pi, n_criteria, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for scenario, assessment, mcda in sorted(zip(scenarios, assessments, mcda_results), key=lambda x: x[2].rank):
    values = [getattr(assessment, c) for c in MCDA_CRITERIA]
    values += values[:1]
    color = type_colors.get(scenario.type.value, "#999")
    ax.plot(angles, values, "o-", linewidth=1.5, color=color, label=f"#{mcda.rank} {scenario.title[:30]}")
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria_labels, fontsize=10)
ax.set_ylim(0, 10.5)
ax.set_title("Scenario Criteria Profiles", fontsize=12, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig("../data/outputs/criteria_radar.png", dpi=150)
plt.show()

# --- 3. AHP Weights Summary ---
print("AHP Criterion Weights:")
print(f"  Consistency Ratio: {ahp.consistency_ratio:.4f} ({'consistent' if ahp.is_consistent else 'INCONSISTENT'})\n")
for criterion, weight in sorted(zip(ahp.criteria, ahp.weights), key=lambda x: -x[1]):
    bar = "#" * int(weight * 50)
    print(f"  {criterion:<16s} {weight:.3f}  {bar}")

# --- 4. Impact vs Probability Matrix (with MCDA rank) ---
fig, ax = plt.subplots(figsize=(10, 8))
mcda_by_id = {r.scenario_id: r for r in mcda_results}

for scenario, assessment in zip(scenarios, assessments):
    color = type_colors.get(scenario.type.value, "#999999")
    size = assessment.confidence * 300 + 50
    alpha = 0.4 + assessment.confidence * 0.5
    mcda = mcda_by_id[scenario.id]

    ax.scatter(assessment.probability, assessment.impact, s=size, c=color,
               alpha=alpha, edgecolors="black", linewidth=0.5, zorder=5)
    ax.annotate(f"#{mcda.rank} {scenario.title[:25]}", (assessment.probability, assessment.impact),
                textcoords="offset points", xytext=(10, 5), fontsize=8,
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

ax.set_xlim(0, 10.5)
ax.set_ylim(0, 10.5)
ax.set_xlabel("Probability", fontsize=12)
ax.set_ylabel("Impact", fontsize=12)
ax.set_title("Impact vs Probability (size = confidence, label = MCDA rank)", fontsize=11)
ax.axhline(y=5, color="gray", linestyle="--", alpha=0.3)
ax.axvline(x=5, color="gray", linestyle="--", alpha=0.3)
ax.text(7.5, 8, "HIGH PRIORITY", ha="center", fontsize=10, color="red", alpha=0.4)
ax.text(2.5, 8, "MONITOR", ha="center", fontsize=10, color="orange", alpha=0.4)
ax.text(7.5, 2, "EXPECTED", ha="center", fontsize=10, color="green", alpha=0.4)
ax.text(2.5, 2, "LOW PRIORITY", ha="center", fontsize=10, color="gray", alpha=0.4)

from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10, label=t.capitalize())
                   for t, c in type_colors.items()]
ax.legend(handles=legend_elements, loc="lower left", fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("../data/outputs/scenario_matrix.png", dpi=150)
plt.show()

## Traceability Report

In [ ]:
report = generate_traceability_report(scenarios, assessments, driver_by_id, kb_state, mcda_results)
print(report)

# save final state with MCDA
final_state = {
    "assessments": [a.model_dump(mode="json") for a in assessments],
    "scenarios": [s.model_dump(mode="json") for s in scenarios],
    "mcda": {
        "ahp_weights": ahp.model_dump(mode="json"),
        "rankings": [r.model_dump(mode="json") for r in mcda_results],
    },
}
with open("../data/outputs/final_analysis.json", "w") as f:
    json.dump(final_state, f, indent=2)
print(f"\nSaved final analysis with MCDA to final_analysis.json")